# Extract Transcript + MoM dari Video Rapat (jalan di Google Colab, tanpa install apa pun di laptop)

**Cara pakai (tidak perlu paham koding):**
1. Klik menu **Runtime > Run all** (atau `Ctrl+F9`).
2. Saat diminta, upload file video rapat Anda.
3. Tunggu sampai semua sel selesai jalan (ada tanda ✔ di kiri tiap sel).
4. Di akhir, file `transcript.md` akan otomatis terdownload ke laptop Anda.
5. Bawa file `transcript.md` itu ke chat Claude Anda untuk dibuatkan ringkasan MoM.

**Video tidak disimpan permanen di mana pun** — hanya diproses sementara di komputer virtual Google
Colab selama sesi ini berjalan, dan hilang begitu tab ditutup / sesi berakhir.

Semua transkripsi berjalan **lokal di server Colab** (model Whisper open-source), jadi tidak butuh
API key OpenAI/Anthropic untuk bagian ini. Speaker ID (siapa bicara) opsional, butuh token Hugging Face
gratis. OCR teks layar (untuk video dengan screen-share) juga opsional.

### (Opsional tapi disarankan) Aktifkan GPU biar lebih cepat
Menu **Runtime > Change runtime type > Hardware accelerator > GPU (T4)**, lalu Save. Tanpa GPU tetap
bisa jalan (pakai CPU), hanya lebih lambat untuk video panjang.

In [1]:
#@title 1. Install kebutuhan (sekali per sesi, sekitar 2-3 menit)
!git clone -q -b claude/relaxed-goldberg-m9mpea https://github.com/muhammad-afifani/Export-Transcript-From-Video.git repo
%cd repo
!apt-get -qq update && apt-get -qq install -y ffmpeg tesseract-ocr tesseract-ocr-ind
!pip install -q -r requirements-core.txt -r requirements-local-whisper.txt -r requirements-ocr.txt
print('Selesai install.')

/content/repo
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-ind.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ind_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-ind (1:4.1.0-2) ...
Setting up tesseract-ocr-ind (1:4.1.0-2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 46.1 MB/s eta 0:00:00
Selesai install.


In [2]:
#@title 2. Upload video rapat Anda
from google.colab import files
uploaded = files.upload()
video_filename = next(iter(uploaded.keys()))
print('Video terupload:', video_filename)

Saving HSSE Enviro Meeting Koordinasi.mp4 to HSSE Enviro Meeting Koordinasi.mp4
Video terupload: HSSE Enviro Meeting Koordinasi.mp4


In [3]:
#@title 3. Pengaturan (opsional, boleh dibiarkan default)
language = "id"  #@param {type:"string"}
local_model_size = "small"  #@param ["tiny", "base", "small", "medium", "large-v3"]
aktifkan_speaker_id = False  #@param {type:"boolean"}
hf_token = ""  #@param {type:"string"}
aktifkan_ocr_layar = True  #@param {type:"boolean"}
ocr_interval_detik = 5  #@param {type:"integer"}

if aktifkan_speaker_id and not hf_token:
    print('[PERINGATAN] Speaker ID diaktifkan tapi hf_token kosong - langkah ini akan dilewati otomatis.')
    print('Cara dapat token gratis: https://huggingface.co/settings/tokens')
    print('Lalu accept license di: https://huggingface.co/pyannote/speaker-diarization-3.1')
    print('dan: https://huggingface.co/pyannote/segmentation-3.0')

if aktifkan_speaker_id and hf_token:
    import os
    os.environ['HF_TOKEN'] = hf_token
    !pip install -q -r requirements-diarization.txt

In [4]:
#@title 4. Jalankan proses (extract audio -> transkrip -> [speaker ID] -> [OCR layar])
import os
import shutil

# files.upload() menyimpan ke direktori kerja saat itu (harusnya sudah /content/repo karena %cd di sel 1).
# Jaga-jaga kalau ternyata tersimpan di /content, pindahkan dulu.
if not os.path.exists(video_filename) and os.path.exists(f'/content/{video_filename}'):
    shutil.move(f'/content/{video_filename}', video_filename)
assert os.path.exists(video_filename), f'File video {video_filename} tidak ditemukan, coba upload ulang di sel 2.'

args = f'--video "{video_filename}" --output-dir output --engine local --local-model-size {local_model_size} --language "{language}" --no-mom --ocr-interval {ocr_interval_detik}'
if not aktifkan_speaker_id or not hf_token:
    args += ' --no-diarization'
if not aktifkan_ocr_layar:
    args += ' --no-ocr'

!python main.py {args}

[1/5] Extract audio dari video (lokal, via ffmpeg)...
[2/5] Transkripsi audio lokal (faster-whisper, tanpa API key)...
  Selesai: 1922 segmen transcript.
[3/5] Speaker diarization dilewati (sesuai opsi).
[4/5] Extract teks layar via OCR (lokal, tesseract)...
  Selesai: 372 potongan teks layar unik terdeteksi.
  Transcript tersimpan: output/transcript.md
[5/5] Generate MoM dilewati (sesuai opsi).

=== Selesai ===
Transcript: output/transcript.md
MoM: (tidak dibuat)


In [5]:
#@title 5. Download hasil transcript.md ke laptop Anda
from google.colab import files
files.download('/content/repo/output/transcript.md')
print('Selesai! Bawa file transcript.md ini ke chat Claude Anda untuk dibuatkan MoM.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Selesai! Bawa file transcript.md ini ke chat Claude Anda untuk dibuatkan MoM.


---
### Langkah selanjutnya
Upload file `transcript.md` yang baru terdownload ke chat Claude Code Anda, lalu minta: "buatkan MoM dari
transcript ini". Claude akan membaca isinya dan langsung menyusun Minutes of Meeting terstruktur, tanpa
perlu API key tambahan.